In [1]:
import pandas as pd
import os

work_dir = "/fsx/amira/papyrus_target_prioritization"

# Read the target prioritization CSV
target_df = pd.read_csv(f"{work_dir}/TargetPrioritization_PBS_IDs_Nov2025.csv")

print(f"Loaded {len(target_df)} targets")
print(f"\nColumns: {target_df.columns.tolist()}")


Loaded 56 targets

Columns: ['Gene name', 'Protein name', 'Cortellis', 'ChEMBL', 'Uniprot ID', 'Protein sequence', 'PDB apo', 'PDB holo + peptide', 'PDB holo + SM']


In [2]:
# Display targets
target_df[['Gene name', 'Protein name', 'Uniprot ID', 'ChEMBL']].head(20)


,Gene name,Protein name,Uniprot ID,ChEMBL
0,GLP1R,glucagon like peptide 1 receptor,P43220,+
1,GCGR,glucagon receptor,P47871,+
2,GIPR,gastric inhibitory polypeptide receptor,P48546,+
3,Amylin receptor,calcitonin receptor + RAMP complex (AMY receptor),P30988 + RAMPs,NaN
4,GLP2R,glucagon like peptide 2 receptor,O95838,+
5,CGRP,calcitonin gene-related peptide receptor,Q16602,+
6,CALCR,calcitonin receptor like receptor,P30988,+
7,NPY2R,neuropeptide Y receptor Y2,P49146,+
8,GHSR,growth hormone secretagogue receptor,Q92847,+
9,GH1R,growth hormone 1 receptor,P01241,NaN


In [3]:
# Extract clean UniProt IDs for filtering
def extract_uniprot_ids(uid_string):
    """Extract UniProt IDs from potentially complex strings like 'P30988 + RAMPs'"""
    if pd.isna(uid_string):
        return []
    ids = []
    for part in str(uid_string).replace('+', ' ').replace(',', ' ').split():
        part = part.strip()
        # UniProt IDs start with P, Q, O, or are 6 alphanumeric chars
        if part and len(part) >= 6 and part[0] in 'PQOA':
            ids.append(part)
    return ids

# Get all unique UniProt IDs from targets
all_uniprot_ids = []
for uid in target_df['Uniprot ID']:
    all_uniprot_ids.extend(extract_uniprot_ids(uid))

target_uniprot_ids = set(all_uniprot_ids)
print(f"Unique UniProt IDs for filtering: {len(target_uniprot_ids)}")
print(f"\nIDs: {sorted(target_uniprot_ids)}")


Unique UniProt IDs for filtering: 55

IDs: ['O95838', 'P00734', 'P00747', 'P01241', 'P03951', 'P03952', 'P04578', 'P04585', 'P05106', 'P08514', 'P08709', 'P09619', 'P11362', 'P12497', 'P22888', 'P23945', 'P24394', 'P30411', 'P30518', 'P30550', 'P30559', 'P30872', 'P30874', 'P30968', 'P30988', 'P31391', 'P32239', 'P32241', 'P32245', 'P32745', 'P33032', 'P33478', 'P35346', 'P35372', 'P35968', 'P37288', 'P41587', 'P41968', 'P43220', 'P47871', 'P47901', 'P48546', 'P49146', 'Q01718', 'Q01726', 'Q03431', 'Q07493', 'Q13564', 'Q16602', 'Q5VWK5', 'Q8NBP7', 'Q8TDV5', 'Q92847', 'Q99705', 'Q9WMX2']


In [5]:
# =============================================================================
# PAPYRUS FILTERING WORKFLOW (SIMPLIFIED)
# =============================================================================
# Directly filter bioactivity file by 'accession' column matching UniProt IDs
# =============================================================================

# Configuration
PAPYRUS_DIR = "/fsx/amira/papyrus_50.5/"
PAPYRUS_BIOACTIVITY = f"{PAPYRUS_DIR}/05.5_combined_set_without_stereochemistry.tsv.xz"

print("=" * 60)
print("PAPYRUS FILTERING - Direct Accession Match")
print("=" * 60)
print(f"\nBioactivity file: {PAPYRUS_BIOACTIVITY}")
print(f"UniProt IDs to filter: {len(target_uniprot_ids)}")


PAPYRUS FILTERING - Direct Accession Match

Bioactivity file: /fsx/amira/papyrus_50.5//05.5_combined_set_without_stereochemistry.tsv.xz
UniProt IDs to filter: 55


In [6]:
# =============================================================================
# STEP 1: Read Bioactivity Data and Filter by Accession (UniProt ID)
# =============================================================================
# The bioactivity file has an 'accession' column with UniProt IDs

print("Reading and filtering Papyrus bioactivity data...")
print(f"Filtering by accession matching: {len(target_uniprot_ids)} UniProt IDs")
print("(This may take several minutes for the large compressed file...)\n")

chunks = []
chunk_size = 500_000
total_rows = 0
total_matches = 0

for i, chunk in enumerate(pd.read_csv(
    PAPYRUS_BIOACTIVITY, 
    sep='\t', 
    compression='xz',
    chunksize=chunk_size,
    low_memory=False
)):
    total_rows += len(chunk)
    # Filter by accession matching our UniProt IDs
    filtered = chunk[chunk['accession'].isin(target_uniprot_ids)]
    if len(filtered) > 0:
        chunks.append(filtered)
        total_matches += len(filtered)
    print(f"  Chunk {i+1}: {total_rows:,} rows processed, {len(filtered):,} matches (total: {total_matches:,})")

# Combine filtered chunks
if chunks:
    molecules_df = pd.concat(chunks, ignore_index=True)
    print(f"\n✓ Extracted {len(molecules_df):,} bioactivity records")
    print(f"✓ Unique molecules (SMILES): {molecules_df['SMILES'].nunique():,}")
    print(f"✓ Unique targets (accession): {molecules_df['accession'].nunique()}")
    
    # Show which UniProt IDs were found
    found_uniprots = set(molecules_df['accession'].unique())
    missing_uniprots = target_uniprot_ids - found_uniprots
    print(f"\n  Found data for {len(found_uniprots)}/{len(target_uniprot_ids)} UniProt IDs")
    if missing_uniprots:
        print(f"  Missing: {sorted(missing_uniprots)}")
else:
    molecules_df = pd.DataFrame()
    print("\n✗ No matching molecules found")


Reading and filtering Papyrus bioactivity data...
Filtering by accession matching: 55 UniProt IDs
(This may take several minutes for the large compressed file...)

  Chunk 1: 500,000 rows processed, 17,950 matches (total: 17,950)
  Chunk 2: 1,000,000 rows processed, 18,147 matches (total: 36,097)
  Chunk 3: 1,500,000 rows processed, 17,964 matches (total: 54,061)
  Chunk 4: 2,000,000 rows processed, 17,924 matches (total: 71,985)
  Chunk 5: 2,500,000 rows processed, 17,919 matches (total: 89,904)
  Chunk 6: 3,000,000 rows processed, 18,154 matches (total: 108,058)
  Chunk 7: 3,500,000 rows processed, 20,072 matches (total: 128,130)
  Chunk 8: 4,000,000 rows processed, 20,209 matches (total: 148,339)
  Chunk 9: 4,500,000 rows processed, 20,069 matches (total: 168,408)
  Chunk 10: 5,000,000 rows processed, 20,104 matches (total: 188,512)
  Chunk 11: 5,500,000 rows processed, 20,190 matches (total: 208,702)
  Chunk 12: 6,000,000 rows processed, 20,191 matches (total: 228,893)
  Chunk 13: 

In [7]:
# =============================================================================
# STEP 2: Analyze Extracted Data
# =============================================================================

if 'molecules_df' in dir() and len(molecules_df) > 0:
    print("Analysis Summary")
    print("=" * 60)
    
    # Map accession back to gene names from target_df
    accession_to_gene = {}
    for _, row in target_df.iterrows():
        uids = extract_uniprot_ids(row['Uniprot ID'])
        for uid in uids:
            accession_to_gene[uid] = row['Gene name']
    
    molecules_df['Gene'] = molecules_df['accession'].map(accession_to_gene)
    
    # Summary by target
    print("\n📊 Bioactivity counts per target:")
    target_summary = molecules_df.groupby(['accession', 'Gene']).agg({
        'SMILES': 'nunique',
        'pchembl_value_Mean': ['count', 'mean', 'std']
    }).round(2)
    target_summary.columns = ['Unique_Molecules', 'N_Activities', 'Mean_pChEMBL', 'Std_pChEMBL']
    target_summary = target_summary.sort_values('Unique_Molecules', ascending=False)
    print(target_summary)
    
    # Quality distribution
    print("\n\n📊 Data quality distribution:")
    print(molecules_df['Quality'].value_counts())
    
    # pChEMBL value distribution
    print("\n\n📊 pChEMBL value statistics:")
    print(molecules_df['pchembl_value_Mean'].describe())
else:
    print("No molecules data. Run Step 1 first.")


Analysis Summary

📊 Bioactivity counts per target:
                                         Unique_Molecules  N_Activities  \
accession Gene                                                            
Q03431    PTH1R                                    392153         41602   
P32245    MC4R                                     351838          2788   
P30559    OXTR                                     320755           909   
P37288    AVPR1A                                   320300          1087   
P43220    GLP1R                                    301853          1972   
P00734    F2                                       220327          6309   
P03951     F11                                     219311          1187   
P49146    NPY2R                                    215525           715   
P35968    KDR/VEGFR2                                10526         10523   
P35372    OPRM1                                      7546          5659   
Q99705    MCHR1                                  

In [8]:
# =============================================================================
# STEP 3: Save Filtered Data
# =============================================================================

if 'molecules_df' in dir() and len(molecules_df) > 0:
    print(f"Saving filtered data to {work_dir}...")
    
    # Save full bioactivity data
    output_bioactivity = f"{work_dir}/papyrus_filtered.tsv.xz"
    molecules_df.to_csv(output_bioactivity, sep='\t', index=False, compression='xz')
    print(f"  ✓ Saved {len(molecules_df):,} bioactivity records to {output_bioactivity}")
    
    
    # Save summary per target
    summary_per_target = molecules_df.groupby(['accession', 'Gene']).agg({
        'SMILES': 'nunique',
        'pchembl_value_Mean': ['count', 'mean']
    }).reset_index()
    summary_per_target.columns = ['UniProtID', 'Gene', 'Unique_Molecules', 'N_Activities', 'Mean_pChEMBL']
    output_summary = f"{work_dir}/papyrus_filtered_summary.tsv"
    summary_per_target.to_csv(output_summary, sep='\t', index=False)
    print(f"  ✓ Saved target summary to {output_summary}")
    
    print("\n" + "=" * 60)
    print("DONE! Output files:")
    print(f"  • {output_bioactivity} - Full bioactivity data")
    print(f"  • {output_summary} - Summary per target")
    print("=" * 60)
else:
    print("No data to save. Run previous steps first.")


Saving filtered data to /fsx/amira/papyrus_target_prioritization...
  ✓ Saved 2,395,676 bioactivity records to /fsx/amira/papyrus_target_prioritization/papyrus_filtered.tsv.xz
  ✓ Saved target summary to /fsx/amira/papyrus_target_prioritization/papyrus_filtered_summary.tsv

DONE! Output files:
  • /fsx/amira/papyrus_target_prioritization/papyrus_filtered.tsv.xz - Full bioactivity data
  • /fsx/amira/papyrus_target_prioritization/papyrus_filtered_summary.tsv - Summary per target


In [9]:
# =============================================================================
# STEP 4: Filter to Keep Only Entries with Bioactivity Values
# =============================================================================

if 'molecules_df' in dir() and len(molecules_df) > 0:
    print("Filtering to keep only entries with bioactivity values...")
    print(f"  Before filtering: {len(molecules_df):,} records")
    
    # Filter: keep only rows where pchembl_value_Mean is not NaN
    molecules_with_activity = molecules_df[molecules_df['pchembl_value_Mean'].notna()].copy()
    
    print(f"  After filtering:  {len(molecules_with_activity):,} records")
    print(f"  Removed: {len(molecules_df) - len(molecules_with_activity):,} records without bioactivity values")
    
    # Summary stats
    print(f"\n📊 Filtered data summary:")
    print(f"  Unique molecules (SMILES): {molecules_with_activity['SMILES'].nunique():,}")
    print(f"  Unique targets: {molecules_with_activity['accession'].nunique()}")
    print(f"  pChEMBL range: {molecules_with_activity['pchembl_value_Mean'].min():.2f} - {molecules_with_activity['pchembl_value_Mean'].max():.2f}")
    
    # Save filtered data with bioactivity values
    print("\n💾 Saving filtered data with bioactivity values...")
    
    # 1. Save bioactivity data (only with values)
    output_active = f"{work_dir}/papyrus_filtered_with_activity.tsv.xz"
    molecules_with_activity.to_csv(output_active, sep='\t', index=False, compression='xz')
    print(f"  ✓ Saved {len(molecules_with_activity):,} records to {output_active}")
    
    # 2. Save SMILES list (all SMILES, including duplicates)
    output_smiles_active = f"{work_dir}/papyrus_filtered_smiles_with_activity.txt"
    molecules_with_activity['SMILES'].to_csv(output_smiles_active, index=False, header=False)
    print(f"  ✓ Saved {len(molecules_with_activity):,} SMILES to {output_smiles_active}")
    
    # 4. Summary per target (with activity only)
    summary_active = molecules_with_activity.groupby(['accession', 'Gene']).agg({
        'SMILES': 'nunique',
        'pchembl_value_Mean': ['count', 'mean', 'std', 'min', 'max']
    }).round(2)
    summary_active.columns = ['Unique_Molecules', 'N_Activities', 'Mean_pChEMBL', 'Std_pChEMBL', 'Min_pChEMBL', 'Max_pChEMBL']
    summary_active = summary_active.sort_values('Unique_Molecules', ascending=False)
    summary_active.to_csv(f"{work_dir}/papyrus_filtered_summary_with_activity.tsv", sep='\t')
    print(f"  ✓ Saved summary to papyrus_filtered_summary_with_activity.tsv")
    
    print("\n" + "=" * 60)
    print("Summary by target (with activity values only):")
    print("=" * 60)
    print(summary_active)
else:
    print("No data. Run previous steps first.")


Filtering to keep only entries with bioactivity values...
  Before filtering: 2,395,676 records
  After filtering:  106,073 records
  Removed: 2,289,603 records without bioactivity values

📊 Filtered data summary:
  Unique molecules (SMILES): 92,948
  Unique targets: 49
  pChEMBL range: 3.02 - 13.00

💾 Saving filtered data with bioactivity values...
  ✓ Saved 106,073 records to /fsx/amira/papyrus_target_prioritization/papyrus_filtered_with_activity.tsv.xz
  ✓ Saved 106,073 SMILES to /fsx/amira/papyrus_target_prioritization/papyrus_filtered_smiles_with_activity.txt
  ✓ Saved summary to papyrus_filtered_summary_with_activity.tsv

Summary by target (with activity values only):
                                         Unique_Molecules  N_Activities  \
accession Gene                                                            
Q03431    PTH1R                                     41602         41602   
P35968    KDR/VEGFR2                                10523         10523   
P00734    F2     

In [10]:
for col in molecules_df.columns:
    print(col)
    print(molecules_df[col].value_counts())

Activity_ID
Activity_ID
AAABHMIRDIOYOK_on_P43220_WT    1
AAAHTCZONLHYSA_on_P09619_WT    1
AAAHTCZONLHYSA_on_P35968_WT    1
AAAOTGDHZDZLFN_on_P30968_WT    1
AAAXBDFZRXKRPL_on_P35968_WT    1
                              ..
ZZZULHVUJCLZBX_on_P43220_WT    1
ZZZXMSLRHJTYGL_on_P30559_WT    1
ZZZXMSLRHJTYGL_on_P32245_WT    1
ZZZXMSLRHJTYGL_on_P37288_WT    1
ZZZXMSLRHJTYGL_on_Q03431_WT    1
Name: count, Length: 2395676, dtype: int64
Quality
Quality
Low       2304779
High        76334
Medium      14563
Name: count, dtype: int64
source
source
ExCAPE-DB                                                                                                                                                                                                                                                                       2334664
ChEMBL30                                                                                                                                                                            